# 📰 Automated Fake News Detection Using ML and NLP
## University of the West of Scotland — MSc Information Technology
**Student:** James Ebukeley Forson  |  **Banner ID:** B01821326

---
### Dataset Files Required
Upload these to `/content/` before running:
| File | Label | Description |
|---|---|---|
| `BBC News Train.csv` | REAL | 1,490 BBC articles (training) |
| `BBC News Test.csv` | REAL | 735 BBC articles (test, needs solution) |
| `BBC News Sample Solution.csv` | — | Category labels for BBC Test |
| `True.csv` | REAL | Additional verified news articles |
| `Fake.csv` | FAKE | Fabricated/misleading articles |

### NLP Pipeline
Text → Lowercase → Remove URLs/punctuation/stopwords → **TF-IDF** (unigrams + bigrams) → **Naïve Bayes / Logistic Regression / LinearSVC**

### Evaluation Metrics
Accuracy · Precision · Recall · F1-Score · AUC-ROC · **5-Fold Cross-Validation**

### Visualisations
ROC Curves · Precision-Recall Curves · K-Fold Box Plot · Top TF-IDF Terms · Confusion Matrix Heatmap · Radar Chart · Stacked Bar

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ═══════════════════════════════════════════════════════
!pip install scikit-learn pandas numpy matplotlib seaborn wordcloud -q
print('✅ Dependencies ready')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2 — Imports
# ═══════════════════════════════════════════════════════
import os, re, string, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
import seaborn as sns
from matplotlib.gridspec import GridSpec

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)
from sklearn.preprocessing import LabelEncoder

# Styling
plt.rcParams.update({
    'figure.facecolor': '#0A0E1A',
    'axes.facecolor':   '#1E2638',
    'axes.edgecolor':   '#2D3748',
    'axes.labelcolor':  '#64748B',
    'xtick.color':      '#64748B',
    'ytick.color':      '#64748B',
    'text.color':       '#E2E8F0',
    'grid.color':       '#2D3748',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})

C_REAL   = '#10B981'
C_FAKE   = '#EF4444'
C_BLUE   = '#3B82F6'
C_GOLD   = '#F59E0B'
C_PURPLE = '#8B5CF6'
MODEL_COLOURS = [C_BLUE, C_REAL, C_GOLD]
print('✅ Imports complete')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3 — Text Preprocessing
# ═══════════════════════════════════════════════════════
STOP_WORDS = ENGLISH_STOP_WORDS

def preprocess(text: str) -> str:
    """
    NLP preprocessing pipeline:
    Lowercase → remove URLs/emails → remove punctuation →
    remove digits → remove stopwords → remove short tokens
    """
    if pd.isna(text) or not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' ', text)
    tokens = [t for t in text.split() if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

# Quick test
sample = 'The Government has announced a NEW policy today! Visit http://gov.uk for 2025 updates.'
print('Original:', sample)
print('Cleaned: ', preprocess(sample))

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 4 — Load & Merge all 5 datasets
# ═══════════════════════════════════════════════════════
LABEL_REAL = 'REAL'
LABEL_FAKE = 'FAKE'

def load_bbc(train_path, test_path, solution_path):
    train = pd.read_csv(train_path)
    train['label'] = LABEL_REAL
    train = train.rename(columns={'Text': 'text', 'Category': 'category'})

    test = pd.read_csv(test_path)
    sol  = pd.read_csv(solution_path)
    test = test.merge(sol, on='ArticleId', how='left')
    test['label'] = LABEL_REAL
    test = test.rename(columns={'Text': 'text', 'Category': 'category'})

    bbc = pd.concat([train, test], ignore_index=True)
    return bbc[['text', 'label', 'category']]

def load_kaggle(true_path, fake_path):
    frames = []
    for path, lbl in [(true_path, LABEL_REAL), (fake_path, LABEL_FAKE)]:
        if os.path.exists(path):
            df = pd.read_csv(path, low_memory=False)
            # Find text column
            tcol = next((c for c in df.columns if c.lower() in ['text','content','body']), None)
            if tcol is None:
                str_cols = [c for c in df.columns if df[c].dtype == object]
                tcol = max(str_cols, key=lambda c: df[c].astype(str).str.len().mean())
            df2 = pd.DataFrame()
            df2['text']     = df[tcol].astype(str)
            df2['label']    = lbl
            df2['category'] = 'unknown'
            frames.append(df2)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ── Paths ──
BBC_TRAIN    = '/content/BBC News Train.csv'
BBC_TEST     = '/content/BBC News Test.csv'
BBC_SOLUTION = '/content/BBC News Sample Solution.csv'
TRUE_PATH    = '/content/True.csv'
FAKE_PATH    = '/content/Fake.csv'

# ── Load ──
bbc_df   = load_bbc(BBC_TRAIN, BBC_TEST, BBC_SOLUTION)
kaggle_df = load_kaggle(TRUE_PATH, FAKE_PATH)

# ── Merge ──
all_frames = [f for f in [bbc_df, kaggle_df] if f is not None and len(f) > 0]
df = pd.concat(all_frames, ignore_index=True)
df = df[df['label'].isin([LABEL_REAL, LABEL_FAKE])]
df = df[df['text'].str.strip().str.len() > 20]
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Total articles: {len(df):,}')
print(df['label'].value_counts())
print('\nCategories (BBC):', df[df['category']!='unknown']['category'].value_counts().to_dict())
display(df.head(4))

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 5 — Preprocessing & TF-IDF Vectorisation
# ═══════════════════════════════════════════════════════
print('Preprocessing texts...')
df['clean_text'] = df['text'].apply(preprocess)

# Remove empty
df = df[df['clean_text'].str.len() > 5].reset_index(drop=True)
print(f'After preprocessing: {len(df):,} articles')

# TF-IDF Vectorisation
MAX_FEATURES = 10000
NGRAM_RANGE  = (1, 2)   # unigrams + bigrams

vectorizer = TfidfVectorizer(
    max_features = MAX_FEATURES,
    ngram_range  = NGRAM_RANGE,
    sublinear_tf = True,    # log(1+tf) — reduces dominance of frequent terms
    min_df       = 2,       # ignore terms in fewer than 2 docs
    strip_accents = 'unicode',
    analyzer     = 'word',
)

le = LabelEncoder()
X = vectorizer.fit_transform(df['clean_text'])
y = le.fit_transform(df['label'])   # FAKE=0, REAL=1

print(f'Feature matrix: {X.shape}  (articles × TF-IDF features)')
print(f'Classes: {le.classes_}  →  {le.transform(le.classes_)}')
print(f'Top 10 features: {list(vectorizer.get_feature_names_out()[:10])}')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 6 — Train / Test Split
# ═══════════════════════════════════════════════════════
TEST_SIZE = 0.2   # 80% train / 20% test

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=42)

print(f'Training:  {X_train.shape[0]:,} articles  |  '
      f'REAL: {int((y_train==1).sum()):,}  FAKE: {int((y_train==0).sum()):,}')
print(f'Test:      {X_test.shape[0]:,} articles   |  '
      f'REAL: {int((y_test==1).sum()):,}  FAKE: {int((y_test==0).sum()):,}')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 7 — Train all 3 Models + 5-Fold CV
# ═══════════════════════════════════════════════════════
MODELS = {
    'Naïve Bayes':          MultinomialNB(alpha=0.1),
    'Logistic Regression':  LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42),
    'SVM (LinearSVC)':      LinearSVC(C=1.0, max_iter=2000, random_state=42),
}

CV_FOLDS  = 5
results   = {}
cv_scores = {}
trained   = {}

for name, model in MODELS.items():
    print(f'Training {name}...')
    m = model.__class__(**model.get_params())
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)

    # Probability calibration (for models without predict_proba)
    try:
        if hasattr(m, 'predict_proba'):
            y_prob = m.predict_proba(X_test)[:, 1]
        else:
            cal = CalibratedClassifierCV(model.__class__(**model.get_params()), cv=3)
            cal.fit(X_train, y_train)
            y_prob = cal.predict_proba(X_test)[:, 1]
    except Exception:
        y_prob = y_pred.astype(float)

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pre, rec, _ = precision_recall_curve(y_test, y_prob)
    roc_auc     = auc(fpr, tpr)

    # 5-Fold Cross-Validation on training set
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    cv_f1 = cross_val_score(
        model.__class__(**model.get_params()),
        X_train, y_train, cv=cv, scoring='f1')

    trained[name]   = m
    cv_scores[name] = cv_f1
    results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
        'auc':       roc_auc,
        'cm':        confusion_matrix(y_test, y_pred),
        'y_pred':    y_pred, 'y_prob': y_prob,
        'fpr': fpr, 'tpr': tpr, 'pre': pre, 'rec': rec,
    }
    print(f'  Accuracy={results[name]["accuracy"]:.4f}  '
          f'Precision={results[name]["precision"]:.4f}  '
          f'Recall={results[name]["recall"]:.4f}  '
          f'F1={results[name]["f1"]:.4f}  '
          f'AUC={roc_auc:.4f}  '
          f'CV-F1={cv_f1.mean():.4f}±{cv_f1.std():.4f}')

best_model = max(results, key=lambda k: results[k]['f1'])
print(f'\n🏆 Best model by F1: {best_model}  (F1={results[best_model]["f1"]:.4f})')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 8 — Full Classification Reports
# ═══════════════════════════════════════════════════════
for name, res in results.items():
    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['FAKE','REAL'],
        digits=4))

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 9 — Chart 1: Stacked Bar (Dataset Composition)
# ═══════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0A0E1A')

# Left: Category distribution
ax = axes[0]; ax.set_facecolor('#1E2638')
cat_counts = df[df['category']!='unknown']['category'].value_counts()
bars = ax.barh(cat_counts.index.tolist(), cat_counts.values,
               color=[C_BLUE, C_REAL, C_GOLD, C_PURPLE, C_FAKE], alpha=0.85)
for bar, val in zip(bars, cat_counts.values):
    ax.text(val+10, bar.get_y()+bar.get_height()/2, f'{val:,}',
            va='center', fontsize=9, color='#E2E8F0', fontweight='bold')
ax.set_xlabel('Article Count', fontsize=9)
ax.set_title('BBC News — Articles by Category\n(All labelled REAL)', fontsize=11, pad=8)
for sp in ax.spines.values(): sp.set_color('#2D3748')

# Right: REAL vs FAKE total
ax2 = axes[1]; ax2.set_facecolor('#1E2638')
lbl_counts = df['label'].value_counts()
colours    = [C_REAL if l=='REAL' else C_FAKE for l in lbl_counts.index]
bars2 = ax2.bar(lbl_counts.index.tolist(), lbl_counts.values,
                color=colours, alpha=0.9, width=0.5)
for bar, val in zip(bars2, lbl_counts.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
             f'{val:,}\n({val/len(df)*100:.1f}%)',
             ha='center', fontsize=10, color='#E2E8F0', fontweight='bold')
ax2.set_ylabel('Article Count', fontsize=9)
ax2.set_title('Class Distribution\n(Binary Classification)', fontsize=11, pad=8)
for sp in ax2.spines.values(): sp.set_color('#2D3748')

plt.suptitle('Dataset Overview', color='#E2E8F0', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 10 — Chart 2: ROC Curves
# ═══════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#0A0E1A'); ax.set_facecolor('#1E2638')

ax.plot([0,1],[0,1],'--', color='#64748B', linewidth=1.2, label='Random Classifier (AUC=0.50)')
for (name, res), colour in zip(results.items(), MODEL_COLOURS):
    ax.plot(res['fpr'], res['tpr'], lw=2.5, color=colour,
            label=f"{name}  (AUC = {res['auc']:.4f})")
    ax.fill_between(res['fpr'], res['tpr'], alpha=0.07, color=colour)

ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=10)
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=10)
ax.set_title('ROC Curves — All Models', fontsize=13, pad=10)
for sp in ax.spines.values(): sp.set_color('#2D3748')
ax.legend(facecolor='#1E2638', edgecolor='#2D3748',
          labelcolor='#E2E8F0', fontsize=9)
ax.yaxis.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 11 — Chart 3: Precision-Recall Curves
# ═══════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#0A0E1A'); ax.set_facecolor('#1E2638')

for (name, res), colour in zip(results.items(), MODEL_COLOURS):
    ax.plot(res['rec'], res['pre'], lw=2.5, color=colour, label=f"{name}")
    ax.fill_between(res['rec'], res['pre'], alpha=0.07, color=colour)

ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('Recall', fontsize=10)
ax.set_ylabel('Precision', fontsize=10)
ax.set_title('Precision-Recall Curves — All Models', fontsize=13, pad=10)
for sp in ax.spines.values(): sp.set_color('#2D3748')
ax.legend(facecolor='#1E2638', edgecolor='#2D3748',
          labelcolor='#E2E8F0', fontsize=9)
ax.yaxis.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 12 — Chart 4: Confusion Matrix Heatmaps (seaborn)
# ═══════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0A0E1A')

for ax, (name, res) in zip(axes, results.items()):
    cm = res['cm']
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['FAKE','REAL'], yticklabels=['FAKE','REAL'],
        ax=ax, linewidths=0.5, linecolor='#2D3748',
        annot_kws={'size': 14, 'weight': 'bold'},
        cbar=False
    )
    ax.set_facecolor('#1E2638')
    ax.set_xlabel('Predicted Label', fontsize=9)
    ax.set_ylabel('True Label', fontsize=9)
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{name}\nTP={tp} FP={fp} FN={fn} TN={tn}', fontsize=9, pad=6)
    ax.tick_params(colors='#E2E8F0')

plt.suptitle('Confusion Matrices — Heatmaps', color='#E2E8F0', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 13 — Chart 5: K-Fold Cross-Validation Box Plot
# ═══════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0A0E1A'); ax.set_facecolor('#1E2638')

names  = list(cv_scores.keys())
scores = [cv_scores[n] for n in names]

bp = ax.boxplot(scores, patch_artist=True, notch=False,
                medianprops=dict(color='white', linewidth=2.5),
                whiskerprops=dict(color='#64748B'),
                capprops=dict(color='#64748B'),
                flierprops=dict(marker='o', color='#64748B', markersize=5))
for patch, colour in zip(bp['boxes'], MODEL_COLOURS):
    patch.set_facecolor(colour); patch.set_alpha(0.75)

# Overlay individual points
for i, (score_list, colour) in enumerate(zip(scores, MODEL_COLOURS), start=1):
    jitter = np.random.normal(0, 0.04, size=len(score_list))
    ax.scatter(np.full_like(score_list, i) + jitter, score_list,
               color=colour, s=40, zorder=5, alpha=0.9, edgecolors='white', linewidths=0.5)

# Annotate means
for i, (name, s) in enumerate(zip(names, scores), start=1):
    ax.text(i, np.mean(s)+0.002, f'μ={np.mean(s):.4f}', ha='center',
            fontsize=8, color='white', fontweight='bold')

ax.set_xticks(range(1, len(names)+1)); ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel('F1-Score', fontsize=10); ax.set_ylim(0, 1.08)
ax.set_title(f'{CV_FOLDS}-Fold Stratified Cross-Validation F1 Scores', fontsize=13, pad=10)
ax.yaxis.grid(True, alpha=0.3)
for sp in ax.spines.values(): sp.set_color('#2D3748')
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 14 — Chart 6: Top TF-IDF Terms (REAL vs FAKE)
# ═══════════════════════════════════════════════════════
feat_names = vectorizer.get_feature_names_out()

def top_features(X_data, y_data, class_idx, n=15):
    mask = y_data == class_idx
    X_cls = X_data[mask]
    weights = np.asarray(X_cls.mean(axis=0)).flatten()
    top_idx = weights.argsort()[::-1][:n]
    return [(feat_names[i], weights[i]) for i in top_idx]

real_feats = top_features(X_train, y_train, class_idx=1, n=15)  # REAL=1
fake_feats = top_features(X_train, y_train, class_idx=0, n=15)  # FAKE=0

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0A0E1A')

for ax, feats, colour, label in [
    (axes[0], real_feats, C_REAL, 'REAL News'),
    (axes[1], fake_feats, C_FAKE, 'FAKE News')
]:
    ax.set_facecolor('#1E2638')
    terms  = [f[0] for f in feats[::-1]]
    weights = [f[1] for f in feats[::-1]]
    bars = ax.barh(terms, weights, color=colour, alpha=0.85)
    for bar, val in zip(bars, weights):
        ax.text(val+0.0001, bar.get_y()+bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=7.5, color='#E2E8F0')
    ax.set_xlabel('Mean TF-IDF Weight', fontsize=9)
    ax.set_title(f'Top 15 TF-IDF Terms — {label}', fontsize=11, pad=8)
    for sp in ax.spines.values(): sp.set_color('#2D3748')

plt.suptitle('Most Discriminative Words per Class', color='#E2E8F0', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 15 — Chart 7: Radar Chart (Model Comparison)
# ═══════════════════════════════════════════════════════
categories   = ['Accuracy','Precision','Recall','F1-Score','AUC-ROC']
metric_keys  = ['accuracy','precision','recall','f1','auc']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0A0E1A'); ax.set_facecolor('#1E2638')
ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, color='#E2E8F0', size=10)
ax.set_ylim(0,1); ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], color='#64748B', size=7)
ax.grid(color='#2D3748', linestyle='--', linewidth=0.6)
ax.spines['polar'].set_color('#2D3748')

for (name, res), colour in zip(results.items(), MODEL_COLOURS):
    vals = [res.get(k,0) for k in metric_keys] + [res.get('accuracy',0)]
    ax.plot(angles, vals, 'o-', linewidth=2.2, label=name, color=colour, markersize=5)
    ax.fill(angles, vals, alpha=0.12, color=colour)

ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15),
          ncol=3, fontsize=9, facecolor='#111827',
          edgecolor='#2D3748', labelcolor='#E2E8F0')
ax.set_title('Model Performance — Radar Chart', color='#E2E8F0', fontsize=13, pad=20)
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 16 — Chart 8: Grouped Bar — All Metrics
# ═══════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0A0E1A'); ax.set_facecolor('#1E2638')

metric_labels = ['Accuracy','Precision','Recall','F1','AUC-ROC']
metric_cols   = ['accuracy','precision','recall','f1','auc']
bar_colours   = [C_BLUE, C_REAL, C_GOLD, C_PURPLE, C_FAKE]
x = np.arange(len(results)); w = 0.14

for i, (met, col, lbl) in enumerate(zip(metric_cols, bar_colours, metric_labels)):
    vals = [v.get(met,0) for v in results.values()]
    bars = ax.bar(x + i*w, vals, w, label=lbl, color=col, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, color='#E2E8F0')

ax.set_xticks(x + w*2)
ax.set_xticklabels(list(results.keys()), fontsize=9, rotation=5)
ax.set_ylim(0, 1.16); ax.set_ylabel('Score', fontsize=10)
ax.set_title('Model Metrics Comparison — All 3 Classifiers', fontsize=13, pad=10)
ax.legend(facecolor='#1E2638', edgecolor='#2D3748',
          labelcolor='#E2E8F0', fontsize=9, ncol=5)
ax.yaxis.grid(True, alpha=0.3)
for sp in ax.spines.values(): sp.set_color('#2D3748')
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 17 — Live Prediction Function
# ═══════════════════════════════════════════════════════
def predict_article(text: str, verbose: bool = True) -> dict:
    """Classify a news article with all trained models."""
    clean = preprocess(text)
    X_new = vectorizer.transform([clean])
    preds = {}
    for name, m in trained.items():
        pred = le.inverse_transform(m.predict(X_new))[0]
        preds[name] = pred
    votes  = list(preds.values())
    final  = 'REAL' if votes.count('REAL') > votes.count('FAKE') else 'FAKE'
    emoji  = '✅' if final=='REAL' else '🔴'
    if verbose:
        print(f'\n{"="*55}')
        print(f'  INPUT: "{text[:80]}..."')
        print(f'  CLEANED: "{clean[:80]}..."')
        print(f'{"="*55}')
        for name, pred in preds.items():
            e = '✅' if pred=='REAL' else '🔴'
            print(f'  {name:<25} → {e} {pred}')
        print(f'  {"─"*53}')
        print(f'  🏆 FINAL VERDICT (majority vote): {emoji} {final}')
        print(f'{"="*55}')
    return {**preds, 'verdict': final}

# Test with sample articles
predict_article('The British government has announced a new economic policy aimed at reducing inflation and supporting household incomes through targeted tax relief measures.')
predict_article('SHOCKING: Scientists secretly admit the moon landing was completely faked by NASA to win the space race against the Soviet Union!')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 18 — Export Results & Trained Artefacts
# ═══════════════════════════════════════════════════════
import pickle, json

# Save best model + vectorizer
best = max(results, key=lambda k: results[k]['f1'])
with open('/content/best_model.pkl', 'wb') as f:
    pickle.dump({'model': trained[best], 'vectorizer': vectorizer,
                 'label_encoder': le, 'model_name': best}, f)

# Save metrics summary
summary = {name: {k: float(v) for k,v in res.items()
                   if k in ['accuracy','precision','recall','f1','auc']}
           for name, res in results.items()}
with open('/content/model_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Save merged dataset with predictions from best model
df_out = df.copy()
df_out['predicted'] = le.inverse_transform(trained[best].predict(
    vectorizer.transform(df_out['clean_text'].fillna(''))))
df_out['correct']   = df_out['label'] == df_out['predicted']
df_out.to_csv('/content/predictions_output.csv', index=False)

print(f'Saved:')
print(f'  /content/best_model.pkl          ({best})')
print(f'  /content/model_results.json')
print(f'  /content/predictions_output.csv  ({len(df_out):,} rows)')

# Download
try:
    from google.colab import files
    for fname in ['best_model.pkl', 'model_results.json', 'predictions_output.csv']:
        files.download(f'/content/{fname}')
except ImportError:
    print('(Not in Colab — files saved locally)')